In [1]:
import torch
import numpy as np

from dinosaw.wrappers import get_model, ModelTypes, WRAPPER_CHECKPOINTS
from dinosaw.utils import add_custom_font, get_features
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


selected_model = "alibi_coco_dinov2_s"
checkpoint = f"../../models/checkpoints/{WRAPPER_CHECKPOINTS[selected_model]}"
model = get_model(selected_model, DEVICE, True, checkpoint)
models = {selected_model: model}


n_dims = model.embed_dim

2026-07-24 12:12:19 | I | factory.py                 : 152 | Building wrapper 'alibi_coco_dinov2_s' on device cuda:0
2026-07-24 12:12:19 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg_ms.pth', model_conf_path='models', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7fe1912de200>, slope_type='constant', n_reg_tokens=4, metric='euclidean', normalize=True, wrap=True, add_cls=True, jitter_mag=0.0)], dtype=torch.float32)
2026-07-24 12:12:19 | I | modifications.py           :  47 | Removed default pos. embed
2026-07-24 12:12:19 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg_ms.pth
2026-07-24 12:12:19 | I | wrapper.py               

In [3]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

replace_with_random_noise: bool = False

features = []
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')
    feats = get_features(model, img, channel_last=True)
    features.append(feats)

2026-07-24 12:12:19 | I | wrapper.py                 :  92 | Processing image, size: [699, 578]
2026-07-24 12:12:19 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,574,686] -> f: [1,384,41,49]
2026-07-24 12:12:19 | I | wrapper.py                 :  92 | Processing image, size: [597, 596]
2026-07-24 12:12:19 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,588,588] -> f: [1,384,42,42]
2026-07-24 12:12:19 | I | wrapper.py                 :  92 | Processing image, size: [800, 528]
2026-07-24 12:12:19 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,798] -> f: [1,384,37,57]
2026-07-24 12:12:19 | I | wrapper.py                 :  92 | Processing image, size: [631, 512]
2026-07-24 12:12:19 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,504,630] -> f: [1,384,36,45]
2026-07-24 12:12:19 | I | wrapper.py                 :  92 | Processing image, size: [700, 700]
2026-07-24 12:12:19 | I | wrapper.py                 : 1

In [4]:
preview_folder = 'data/linear_probe/'
preview_img_names = ['black_square_518.png', 'bulldog_518.png', 'edinburgh.png', '000.png']
preview_feats = []
preview_imgs: list[Image.Image] = []

for fname in preview_img_names:
    img_path = f'{preview_folder}/{fname}'
    img = Image.open(img_path).convert('RGB')
    img = img.resize((518, 518))
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')
    feats = get_features(model, img, channel_last=True)
    preview_imgs.append(img)
    preview_feats.append(feats)


2026-07-24 12:12:20 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 12:12:20 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 12:12:20 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 12:12:20 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 12:12:20 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 12:12:20 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]
2026-07-24 12:12:20 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 12:12:20 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,518,518] -> f: [1,384,37,37]


In [5]:
ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramps_to_results: dict[RampTypes, list[LinearProbeResult]] = {r: [] for r in ramps}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for ramp in ramps:
    for i in range(n_imgs):
        feats = features[i]
        result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
        ramps_to_results[ramp].append(result)

In [6]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [7]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none', lw=0.25)
    ax.add_collection(pc)

In [8]:
# %%capture
n_rows, n_cols = 4, 9
# TITLE_PAD = 20
# FS = 26


W, H = 7.5, 1  * 2.3
fig = plt.figure(figsize=(W, H))

plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False

add_custom_font('resources/fonts', 'Grotesk')


w_spacing = [1 for _ in range(n_cols)]
SPACE_ROW_IDXS = (4,)

FIG_B_COL_OFFSET = 1
FIG_B_W_COLS = 2
FIG_C_COL_OFFSET = FIG_B_COL_OFFSET + FIG_B_W_COLS + 2

w_spacing[4] = 0.05

w_spacing[FIG_C_COL_OFFSET:] = [0.7] * len(w_spacing[FIG_C_COL_OFFSET:])

# fig = plt.figure(figsize=(W * n_cols, H * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, wspace=0.2, hspace=0.35)



dino_colors: dict[RampTypes, str] = {
    'lr': '#5762D5',
    'ud': '#6370C0',
    'diag': '#6E7DAB',
    'radial': '#575366',
}

alibi_colors: dict[RampTypes, str] = {
    'lr': '#16ce37',
    'ud': '#87f1ab',
    'diag': '#5f978b',
    'radial': '#133b32',
}

vit_b_colors: dict[RampTypes, str] = {
    'lr': '#FFE048',
    'ud': '#CBCA74',
    'diag': '#969765',
    'radial': '#2D3047',
}


model_to_ramp_colours: dict[ModelTypes, dict[RampTypes, str]] = {"dinov2_s": dino_colors, 'alibi_coco_dinov2_s': alibi_colors, 'mae_b': vit_b_colors}


colors = model_to_ramp_colours.get(selected_model, dino_colors)

ramp_to_title: dict[RampTypes, str] = {
    'lr': 'Left-right',
    'ud': 'Up-down',
    'diag': 'Diagonal',
    'radial': 'Radial',
}
# spacer_row = fig.add_subplot(gs[:, 5])


top_left_ramp_ax = None
for row, ramp in enumerate(ramps):
    h, w = 34, 34
    ramp_arr = get_ramp(ramp, h, w)
    ramp_ax = fig.add_subplot(gs[row, 0])
    ramp_ax.imshow(ramp_arr, cmap='viridis', vmin=0, vmax=1)

    mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
    add_red_square_overlay(ramp_ax, mask, 1, 1)

    ramp_ax.set_xticks([])
    ramp_ax.set_yticks([])

    ramp_ax.set_ylabel(ramp_to_title[ramp], fontsize=7)

    if row == 0:
        ramp_ax.set_title('Target ramp',)
        top_left_ramp_ax = ramp_ax


worst_channels = []
for row, ramp in enumerate(ramps):
    ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET:FIG_B_COL_OFFSET + FIG_B_W_COLS])
    mean_channel_scores, std_channel_scores, mean_score, _, mean_pred = average_results(ramps_to_results[ramp])

    print(f"{ramp}: {np.argsort(-mean_channel_scores)[:6]}")
    worst_channels.extend(list(np.argsort(-mean_channel_scores)[:2]))

    ax.hlines(0, 0, n_dims, 'red', '--', lw=0.5)

    ax.plot(mean_channel_scores, color=colors[ramp], lw=0.5)
    ax.fill_between(
        np.arange(n_dims),
        mean_channel_scores - std_channel_scores,
        mean_channel_scores + std_channel_scores,
        color=colors[ramp],
        alpha=0.3,
    )
    ax.set_ylim(-0.1, 1)
    ax.set_xlim(0 -10, n_dims + 10)

    ax.tick_params(axis='both', labelsize=6)
    if row != len(ramps) - 1:
        ax.set_xticklabels([])
        # ax.set_yticklabels([])
    ax.grid(alpha=0.2)

    # ax.ticklabel_format(size=FS - 4)

    mean_pred_ax = fig.add_subplot(gs[row, FIG_B_COL_OFFSET + FIG_B_W_COLS])
    mean_pred_ax.imshow(mean_pred, cmap='viridis', vmin=0, vmax=1)

    if row == 0:
        ax.set_title('Per-channel ' + r'$R^2$' +  'scores')
        mean_pred_ax.set_title('Mean prediction')
    elif row == len(ramps) - 1:
        ax.set_xlabel('Channel',)
        mean_pred_ax.set_xlabel('',)
    

    mean_pred_ax.set_ylabel(f'$R^{2}:${mean_score:.2f}', fontsize=7)
    mean_pred_ax.set_xticks([])
    mean_pred_ax.set_yticks([])

bad_channel_map = {
    'dv2': [55, 89, 228], 
    'dvt': [55, 113, 188],
    'dv2_b': [39, 354, 480], 
    'mae_b': [18, 390, 599], 
    'dv': [293, 61, 89 ], 
    'dv_b': [432, 266, 99 ], 
    'dv3': [149, 123, 74], 
    'clip_b': [69, 526, 554], 
    'eva02': [180, 192, 180], 
    'sam_b': [50, 27, 210],
    'deit': [173, 99, 302],
    'alibi_dv2_cb': [78, 73, 228],
    'alibi_coco_dinov2_s': [118, 228, 359],
}
bad_channels = bad_channel_map.get(selected_model, [47, 117, 359])

top_right_ax = None
for i, img in enumerate(preview_imgs):
    img_ax = fig.add_subplot(gs[i, FIG_C_COL_OFFSET])
    img_ax.imshow(img)
    img_ax.set_xticks([])
    img_ax.set_yticks([])
    if i == 0:
        top_right_ax = img_ax


    for j, ch in enumerate(bad_channels):
        ax = fig.add_subplot(gs[i, FIG_C_COL_OFFSET + j + 1])
        feats = preview_feats[i]
        selected_ch = feats[:, :, ch]
        ax.imshow(selected_ch, cmap='viridis')

        if i == 0:
            # ax.set_title(f'Ch. {ch}', fontsize=FS, pad=TITLE_PAD)
            ax.set_title(f'Ch. {ch}')

        ax.set_xticks([])
        ax.set_yticks([])


labels = ['(a)', '(b)']
axes = [top_left_ramp_ax, top_right_ax]
# print(gs.subplots()[0, 0])
for i in range(2):
    ax = axes[i]
    if i == 0:
        x = -1.1
    else:
        x = -0.3
    text = labels[i]
    ax.text(x, 1.3, text, transform=ax.transAxes,
             fontweight='bold', color='black')



SAVE = True
if SAVE:
    plt.savefig("saved/S5.pdf", dpi=300, bbox_inches='tight')
    plt.close()

findfont: Failed to find font weight normal, now using 300.


findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.


lr: [ 47 228 113 248 183 117]
ud: [228 113  47  73 260 242]
diag: [228  47 117 113 189 169]
radial: [228 113  47 359 331 336]


findfont: Failed to find font weight normal, now using 300.
